In [7]:
# =========================
# Cache helpers
# =========================
import csv, hashlib, time

FEAT_CACHE = "data/feature_cache.csv"  # id,path,mtime,bass_ratio,...,bass_beat_var

def _file_id(path):
    # stable id: sha1 of absolute path (windows-safe)
    p = os.path.abspath(path).encode()
    return hashlib.sha1(p).hexdigest()[:16]

def load_feat_cache():
    os.makedirs(os.path.dirname(FEAT_CACHE), exist_ok=True)
    if not os.path.isfile(FEAT_CACHE): return {}
    out = {}
    with open(FEAT_CACHE, newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        for row in r:
            out[row["id"]] = row
    return out

def save_feat_cache_row(row):
    exists = os.path.isfile(FEAT_CACHE)
    with open(FEAT_CACHE, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["id","path","mtime"]+FEATURE_KEYS)
        if not exists: w.writeheader()
        w.writerow(row)

def extract_feature_vector_cached(path):
    cache = load_feat_cache()
    fid = _file_id(path)
    mtime = str(os.path.getmtime(path))
    if fid in cache and cache[fid].get("mtime")==mtime:
        vals = [float(cache[fid][k]) if cache[fid][k] not in ("", "nan") else np.nan for k in FEATURE_KEYS]
        return np.array(vals, dtype=float)
    feat = analyze_audio_extended(path)
    if feat is None: return None
    row = {"id": fid, "path": os.path.abspath(path), "mtime": mtime}
    row.update({k: ("" if np.isnan(feat[k]) else feat[k]) for k in FEATURE_KEYS})
    save_feat_cache_row(row)
    return np.array([feat[k] for k in FEATURE_KEYS], dtype=float)

In [8]:
EXCLUDE_TOKENS = ("cover","bass cover","tabs","play along","play-along","lesson","tutorial",
                  "backing","karaoke","mashup","loop","mix","full album","remix","reaction")

def should_skip_title(title: str) -> bool:
    t = title.lower()
    return any(tok in t for tok in EXCLUDE_TOKENS)

def existing_ids_in_dir(d):
    ids = set()
    if not os.path.isdir(d): return ids
    for f in os.listdir(d):
        if f.endswith(".wav") and "[" in f and "]" in f:
            ids.add(f.rsplit("[",1)[-1].rsplit("]",1)[0])  # parse [id]
    return ids

def download_youtube_results(keyword, save_dir="data/candidates", max_results=3):
    os.makedirs(save_dir, exist_ok=True)
    neg = "-live -lesson -tutorial -backing -playlist -mix -loop -cover -full album -reaction"
    query = f'ytsearch{max_results}:{keyword} {neg}'
    out_tmpl = os.path.join(save_dir, "%(title)s [%(id)s].%(ext)s")

    # fetch JSON to filter titles & duplicates
    cmd = [
        "yt-dlp", query, "--print-json", "--skip-download",
        "--no-playlist", "--match-filter", "duration>=120 & duration<=420 & !is_live & !was_live"
    ]
    try:
        js = subprocess.run(cmd, capture_output=True, text=True, check=False).stdout.strip().splitlines()
    except Exception as e:
        print(f"[Warn] Metadata fetch failed: {e}"); js = []

    wanted_ids = []
    seen_ids = existing_ids_in_dir(save_dir)
    for line in js:
        try:
            meta = json.loads(line)
            title = meta.get("title","")
            vid = meta.get("id","")
            if not vid or vid in seen_ids: continue
            if should_skip_title(title): continue
            wanted_ids.append(vid)
        except: 
            pass

    if not wanted_ids:
        print(f'No clean results for "{keyword}"'); return

    # actual download by IDs
    for vid in wanted_ids:
        dl = ["yt-dlp", f"https://www.youtube.com/watch?v={vid}",
              "--extract-audio","--audio-format","wav","-o", out_tmpl, "--quiet","--no-warnings"]
        subprocess.run(dl, check=False)


In [9]:
def compare_to_reference(reference_path, candidate_dir):
    feats, names = [], []
    ref_vec = extract_feature_vector_cached(reference_path)
    if ref_vec is None: raise ValueError("Reference track too short or invalid.")
    feats.append(ref_vec); names.append(os.path.basename(reference_path))

    for fname in os.listdir(candidate_dir):
        if not fname.lower().endswith(".wav"): continue
        c_path = os.path.join(candidate_dir, fname)
        c_vec = extract_feature_vector_cached(c_path)
        if c_vec is None: continue
        feats.append(c_vec); names.append(fname)

    X = np.vstack(feats)
    Xz, med, iqr = robust_standardize(X)
    ref_z, cand_z = Xz[0], Xz[1:]
    cand_names = names[1:]

    results = []
    for fname, vec in zip(cand_names, cand_z):
        sim_raw = safe_weighted_cosine(ref_z, vec, WEIGHTS)
        if np.isnan(sim_raw): continue
        sim01 = (sim_raw + 1.0) / 2.0
        results.append((fname, sim_raw, sim01))
    return sorted(results, key=lambda x: -x[1])


In [10]:
def recommend_batch(audio_dir="data/audio_raw", out_csv="data/recs_summary.csv",
                    top_n=5, per_query=2):
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    rows = []
    for f in os.listdir(audio_dir):
        if not f.lower().endswith(".wav"): continue
        ref = os.path.join(audio_dir, f)
        cand_dir = os.path.join("data/candidates", os.path.splitext(f)[0])
        os.makedirs(cand_dir, exist_ok=True)

        print(f"\n=== Reference: {f} ===")
        # 후보 폴더를 날리지 말고 (캐시 재활용), 새 쿼리 결과만 추가 다운로드
        try:
            # 1) query 생성
            feat = analyze_audio_extended(ref)
            if feat is None: 
                print(" skip (too short)"); 
                continue
            queries = generate_queries_from_features(feat)
            for q in queries:
                download_youtube_results(q, save_dir=cand_dir, max_results=per_query)

            # 2) 비교
            ranked = compare_to_reference(ref, cand_dir)[:top_n]
            for title, sim_raw, sim01 in ranked:
                rows.append({"reference": f, "candidate": title, "sim_raw": sim_raw, "sim01": sim01})
                print(f"{title:60s} | {sim01:.3f}")
        except Exception as e:
            print(f" failed: {e}")

    # save summary
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["reference","candidate","sim_raw","sim01"])
        w.writeheader()
        for r in rows: w.writerow(r)
    print(f"\n[Done] Saved summary → {out_csv}")


In [11]:
# 개별
# RUN("./data/audio_raw/Billie_Jean.wav", top_n=5, per_query=2)

# 배치
recommend_batch(audio_dir="data/audio_raw", top_n=5, per_query=2)


=== Reference: Another_One_Bites.wav ===
No clean results for "deep bass groove official audio"
No clean results for "funk bass line full song"
(FREE) Michael Jackson 80s type beat - Starlight Lover [8MZR9cvKURg].wav | 0.636
Miss - Hip Hop [Audio] [zuijRF0bdiY].wav                     | 0.626
TELEFILME - Hi-Fi Ghost [Audio] [-jid6dr145s].wav            | 0.344
His Wishlist： 43. In the Disco was One (Instrumental) [qKej0y7RDy0].wav | 0.283
Got 2 Grind [s4SgL1W6syU].wav                                | 0.195

=== Reference: Billie_Jean.wav ===
No clean results for "deep bass groove official audio"
No clean results for "funk bass line full song"
Miss - Hip Hop [Audio] [zuijRF0bdiY].wav                     | 0.758
TELEFILME - Hi-Fi Ghost [Audio] [-jid6dr145s].wav            | 0.629
Got 2 Grind [s4SgL1W6syU].wav                                | 0.435
(FREE) Michael Jackson 80s type beat - Starlight Lover [8MZR9cvKURg].wav | 0.403
His Wishlist： 43. In the Disco was One (Instrumental) [qKej0